In [ ]:
"""
EfficientPhys rPPG pipeline (HR + RR) 
"""

import os
import sys
import time
import argparse
import threading
import collections
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import cv2
import torch
import torch.nn as nn
import scipy.signal


# =============================================================================
# Model (matches rPPG-Toolbox EfficientPhys; diff + BatchNorm done internally)
# =============================================================================
class Attention_mask(nn.Module):
    def forward(self, x):
        xsum = torch.sum(x, dim=2, keepdim=True)
        xsum = torch.sum(xsum, dim=3, keepdim=True)
        s = x.size()
        return x / (xsum + 1e-8) * s[2] * s[3] * 0.5


class TSM(nn.Module):
    def __init__(self, n_segment=20, fold_div=3):
        super().__init__()
        self.n_segment = n_segment
        self.fold_div = fold_div

    def forward(self, x):
        nt, c, h, w = x.size()
        n_batch = nt // self.n_segment
        x = x.view(n_batch, self.n_segment, c, h, w)
        fold = c // self.fold_div
        out = torch.zeros_like(x)
        out[:, :-1, :fold] = x[:, 1:, :fold]              # shift left  (toward present)
        out[:, 1:, fold:2 * fold] = x[:, :-1, fold:2 * fold]  # shift right (toward future)
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]
        return out.view(nt, c, h, w)


class EfficientPhys(nn.Module):
    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3,
                 dropout_rate1=0.25, dropout_rate2=0.5, pool_size=(2, 2),
                 nb_dense=128, frame_depth=20, img_size=72):
        super().__init__()
        self.TSM_1 = TSM(n_segment=frame_depth)
        self.TSM_2 = TSM(n_segment=frame_depth)
        self.TSM_3 = TSM(n_segment=frame_depth)
        self.TSM_4 = TSM(n_segment=frame_depth)
        self.motion_conv1 = nn.Conv2d(in_channels, nb_filters1, kernel_size, padding=1)
        self.motion_conv2 = nn.Conv2d(nb_filters1, nb_filters1, kernel_size)
        self.motion_conv3 = nn.Conv2d(nb_filters1, nb_filters2, kernel_size, padding=1)
        self.motion_conv4 = nn.Conv2d(nb_filters2, nb_filters2, kernel_size)
        self.apperance_att_conv1 = nn.Conv2d(nb_filters1, 1, kernel_size=1)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(nb_filters2, 1, kernel_size=1)
        self.attn_mask_2 = Attention_mask()
        self.avg_pooling_1 = nn.AvgPool2d(pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(pool_size)
        self.dropout_1 = nn.Dropout(dropout_rate1)
        self.dropout_3 = nn.Dropout(dropout_rate1)
        self.dropout_4 = nn.Dropout(dropout_rate2)
        dense_in = {36: 3136, 72: 16384, 96: 30976}
        if img_size not in dense_in:
            raise ValueError(f"Unsupported img_size {img_size}; expected one of {list(dense_in)}")
        self.final_dense_1 = nn.Linear(dense_in[img_size], nb_dense)
        self.final_dense_2 = nn.Linear(nb_dense, 1)
        self.batch_norm = nn.BatchNorm2d(3)

    def forward(self, inputs):
        inputs = torch.diff(inputs, dim=0)          # (N,3,H,W) -> (N-1,3,H,W)
        inputs = self.batch_norm(inputs)
        x = self.TSM_1(inputs)
        d1 = torch.tanh(self.motion_conv1(x))
        d1 = self.TSM_2(d1)
        d2 = torch.tanh(self.motion_conv2(d1))
        g1 = self.attn_mask_1(torch.sigmoid(self.apperance_att_conv1(d2)))
        d4 = self.dropout_1(self.avg_pooling_1(d2 * g1))
        d4 = self.TSM_3(d4)
        d5 = torch.tanh(self.motion_conv3(d4))
        d5 = self.TSM_4(d5)
        d6 = torch.tanh(self.motion_conv4(d5))
        g2 = self.attn_mask_2(torch.sigmoid(self.apperance_att_conv2(d6)))
        d8 = self.dropout_3(self.avg_pooling_3(d6 * g2))
        d9 = d8.view(d8.size(0), -1)
        d10 = torch.tanh(self.final_dense_1(d9))
        return self.final_dense_2(self.dropout_4(d10))


def load_model(weight_path, device, frame_depth, img_size):
    model = EfficientPhys(frame_depth=frame_depth, img_size=img_size)
    ckpt = torch.load(weight_path, map_location=device)
    if isinstance(ckpt, dict):
        for k in ("state_dict", "model_state_dict"):
            if k in ckpt:
                ckpt = ckpt[k]
                break
    cleaned = {k.replace("module.", ""): v for k, v in ckpt.items()}
    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    # Loud reporting: silent strict=False is how half-initialized models slip through.
    if missing:
        print(f"[WARN] {len(missing)} layers NOT found in checkpoint (random-init!): "
              f"{missing[:6]}{' ...' if len(missing) > 6 else ''}")
    if unexpected:
        print(f"[WARN] {len(unexpected)} checkpoint keys unused: "
              f"{unexpected[:6]}{' ...' if len(unexpected) > 6 else ''}")
    if any('final_dense' in m or 'motion_conv' in m for m in missing):
        print("[ERROR] Core layers were not loaded — check the checkpoint / architecture match.")
    model.to(device).eval()
    return model


# =============================================================================
# Face ROI  (Haar by default)
# =============================================================================
class FaceROI:
    def __init__(self, use_face=True, smooth=0.6, pad=0.20):
        self.use_face = use_face
        self.smooth = smooth
        self.pad = pad
        self.box = None
        self.det = None
        if use_face:
            p = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
            self.det = cv2.CascadeClassifier(p)
            if self.det.empty():
                print("[WARN] Haar cascade failed to load; falling back to center crop.")
                self.use_face = False

    def _center_box(self, frame):
        h, w = frame.shape[:2]
        s = int(min(h, w) * 0.6)
        cx, cy = w // 2, h // 2
        return [cx - s // 2, cy - s // 2, s, s]

    def get(self, frame):
        h, w = frame.shape[:2]
        box = None
        if self.use_face:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = self.det.detectMultiScale(gray, 1.2, 5, minSize=(80, 80))
            if len(faces):
                box = list(max(faces, key=lambda b: b[2] * b[3]))  # largest face
        if box is None:
            box = self.box if self.box is not None else self._center_box(frame)
        # exponential smoothing to suppress jitter (jitter -> motion artifacts)
        if self.box is None:
            self.box = box
        else:
            a = self.smooth
            self.box = [a * o + (1 - a) * n for o, n in zip(self.box, box)]
        x, y, bw, bh = self.box
        # pad and square-ish the box, then clamp to frame
        px, py = bw * self.pad, bh * self.pad
        x1 = int(max(x - px, 0)); y1 = int(max(y - py, 0))
        x2 = int(min(x + bw + px, w)); y2 = int(min(y + bh + py, h))
        return x1, y1, x2, y2


def crop_resize_rgb(frame, box, img_size):
    x1, y1, x2, y2 = box
    roi = frame[y1:y2, x1:x2]
    if roi.size == 0:
        roi = frame
    roi = cv2.resize(roi, (img_size, img_size), interpolation=cv2.INTER_AREA)
    roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0  # [0,1]
    return np.transpose(roi, (2, 0, 1))                                    # (3,H,W)


# =============================================================================
# Inference
# =============================================================================
def _standardize(clip):
    return (clip - clip.mean()) / (clip.std() + 1e-8)


def infer_bvp(model, frames_chw, device, frame_depth, data_type, chunk_blocks=20):
    """frames_chw: (N,3,H,W) float32 in [0,1]. Returns BVP of length ~N-1.

    diff() reduces N->N-1 and TSM needs (N-1) divisible by frame_depth, so we feed
    blocks whose length-1 is a multiple of frame_depth and overlap blocks by 1 frame
    to keep the output waveform contiguous (this mirrors how the toolbox chunks data).
    """
    N = frames_chw.shape[0]
    if N < frame_depth + 1:
        return None
    clip_all = _standardize(frames_chw) if data_type == "standardized" else frames_chw
    BS = chunk_blocks * frame_depth + 1
    outs, start = [], 0
    while start + frame_depth + 1 <= N:
        L = min(BS, N - start)
        M = ((L - 1) // frame_depth) * frame_depth          # largest valid diff-count
        if M < frame_depth:
            break
        block = np.ascontiguousarray(clip_all[start:start + M + 1])
        x = torch.from_numpy(block).float().to(device)
        with torch.no_grad():
            o = model(x).detach().cpu().numpy().reshape(-1)  # M outputs
        outs.append(o)
        start += M                                           # overlap by 1 frame
    return np.concatenate(outs) if outs else None


# =============================================================================
# Signal processing
# =============================================================================
def resample_uniform(sig, t, fs_target):
    """Non-uniform samples (t) -> uniform grid at fs_target. fs rPPG assumes uniform sampling."""
    t = np.asarray(t, float)
    grid = np.arange(t[0], t[-1], 1.0 / fs_target)
    return np.interp(grid, t, sig), fs_target


def bandpass(sig, fs, lo, hi, order=2):
    nyq  = 0.5 * fs
    b, a = scipy.signal.butter(order, [lo / nyq, hi / nyq], btype="band")
    padlen = 3 * max(len(a), len(b))
    if len(sig) <= padlen:
        return sig - np.mean(sig)
    return scipy.signal.filtfilt(b, a, sig)


def _parabolic(f, p, i):
    """Sub-bin peak refinement around discrete index i."""
    if i <= 0 or i >= len(p) - 1:
        return f[i]
    y0, y1, y2 = p[i - 1], p[i], p[i + 1]
    denom = (y0 - 2 * y1 + y2)
    off = 0.5 * (y0 - y2) / denom if denom != 0 else 0.0
    return f[i] + off * (f[1] - f[0])


def fft_rate_snr(sig, fs, band, harmonic_for_snr=True):
    """Returns (rate_per_min, snr_dB). Zero-pads for fine bins + parabolic interpolation."""
    sig = np.asarray(sig, float)
    sig = sig - sig.mean()
    if len(sig) < fs * 4:
        return None, None
    nfft = int(2 ** np.ceil(np.log2(max(len(sig), fs * 30))))  # >= ~30s equivalent resolution
    f, p = scipy.signal.periodogram(sig, fs=fs, window="hann", nfft=nfft, detrend=False)
    m = (f >= band[0]) & (f <= band[1])
    if not np.any(m):
        return None, None
    idx = np.where(m)[0][np.argmax(p[m])]
    f_peak = _parabolic(f, p, idx)
    rate = f_peak * 60.0
    # SNR: power near peak (+harmonic) vs the rest of the band
    bw = 0.15
    sig_mask = (np.abs(f - f_peak) <= bw)
    if harmonic_for_snr:
        sig_mask |= (np.abs(f - 2 * f_peak) <= bw)
    sig_pow = np.sum(p[sig_mask & m])
    noise_pow = np.sum(p[m & ~sig_mask]) + 1e-12
    snr = 10 * np.log10(sig_pow / noise_pow + 1e-12)
    return rate, snr


def estimate_hr(bvp_u, fs, band=(0.7, 3.0)):
    return fft_rate_snr(bandpass(bvp_u, fs, *band), fs, band)


def estimate_rr(bvp_u, fs, hr_band=(0.7, 3.0), rr_band=(0.1, 0.5), min_sec=20):
    """Respiration via amplitude-envelope (RIAV) of the pulse more robust than the raw
    low-frequency band, because EfficientPhys output is derivative-like and high-pass-biased,
    which suppresses direct low-frequency (respiratory) content. Still only approximate:
    the model was not trained for respiration. Needs a long window (>= ~20-30 s)."""
    if len(bvp_u) < fs * min_sec:
        return None, None
    pulse = bandpass(bvp_u, fs, *hr_band)
    env = np.abs(scipy.signal.hilbert(pulse))   # amplitude envelope
    env = bandpass(env - env.mean(), fs, *rr_band, order=2)
    return fft_rate_snr(env, fs, rr_band, harmonic_for_snr=False)


# =============================================================================
# Webcam capture 
# =============================================================================
class FreshestFrame(threading.Thread):
    def __init__(self, cap):
        super().__init__(daemon=True)
        self.cap = cap
        self.lock = threading.Lock()
        self.frame = None
        self.ts = None
        self.running = True
        self.start()

    def run(self):
        while self.running:
            ok, f = self.cap.read()
            if not ok:
                self.running = False
                break
            with self.lock:
                self.frame, self.ts = f, time.time()

    def read(self):
        with self.lock:
            return (self.frame.copy(), self.ts) if self.frame is not None else (None, None)

    def stop(self):
        self.running = False
        self.cap.release()


def draw_waveform(canvas, sig, x, y, w, h, color):
    if sig is None or len(sig) < 2:
        return
    s = np.asarray(sig[-w:], float)
    s = s - s.min()
    rng = s.max() if s.max() > 1e-9 else 1.0
    s = s / rng
    pts = [(x + i, int(y + h - s_i * h)) for i, s_i in enumerate(s[-w:])]
    cv2.polylines(canvas, [np.array(pts, np.int32)], False, color, 1)


# =============================================================================
# Modes
# =============================================================================
def run_webcam(args, model, device):
    cap = cv2.VideoCapture(int(args.source))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open webcam {args.source}")
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    grab = FreshestFrame(cap)
    roi = FaceROI(use_face=not args.no_face)

    win = max(args.hr_window, args.rr_window)
    frame_buf = collections.deque(maxlen=args.fps_hint * win + 10)
    ts_buf = collections.deque(maxlen=args.fps_hint * win + 10)
    interval_buf = collections.deque(maxlen=120)

    hr = rr = hr_snr = rr_snr = None
    last_box = (0, 0, 1, 1)
    last_infer = 0.0
    bvp_disp = None
    prev_ts = None
    print(f"[info] webcam mode on {device}; press q/esc to quit.")

    while True:
        frame, ts = grab.read()
        if frame is None:
            if not grab.running:
                break
            time.sleep(0.005)
            continue
        if prev_ts is not None and ts > prev_ts:
            interval_buf.append(ts - prev_ts)
        prev_ts = ts

        last_box = roi.get(frame)
        frame_buf.append(crop_resize_rgb(frame, last_box, args.img_size))
        ts_buf.append(ts)

        now = time.time()
        if now - last_infer > args.update_sec and len(frame_buf) >= args.fps_hint * 4:
            fs_u = float(np.clip(1.0 / np.median(interval_buf), 5, 60)) if interval_buf else args.fps_hint
            frames = np.stack(frame_buf, 0)
            bvp = infer_bvp(model, frames, device, args.frame_depth, args.data_type)
            if bvp is not None and len(bvp) > args.frame_depth:
                out_ts = np.asarray(ts_buf)[1:1 + len(bvp)]
                bvp_u, fs_r = resample_uniform(bvp, out_ts, fs_u)
                bvp_disp = bandpass(bvp_u, fs_r, 0.7, 3.0)
                hr, hr_snr = estimate_hr(bvp_u, fs_r)
                if len(bvp_u) >= fs_r * args.rr_window * 0.8:
                    rr, rr_snr = estimate_rr(bvp_u, fs_r)
            last_infer = now

        disp = frame.copy()
        x1, y1, x2, y2 = last_box
        cv2.rectangle(disp, (x1, y1), (x2, y2), (0, 255, 0), 2)

        def fmt(label, val, snr, unit):
            if val is None:
                return f"{label}: --"
            tag = "" if (snr is None) else ("  (low conf)" if snr < args.snr_min else "")
            return f"{label}: {val:4.1f} {unit}{tag}"

        cv2.putText(disp, fmt("HR", hr, hr_snr, "bpm"), (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(disp, fmt("RR", rr, rr_snr, "rpm") + " (approx)", (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 200, 0), 2)
        fps_now = 1.0 / np.median(interval_buf) if interval_buf else 0
        cv2.putText(disp, f"fps~{fps_now:4.1f}", (20, disp.shape[0] - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        draw_waveform(disp, bvp_disp, 20, disp.shape[0] - 120, min(360, disp.shape[1] - 40),
                      70, (0, 255, 0))

        cv2.imshow("EfficientPhys HR/RR", disp)
        if (cv2.waitKey(1) & 0xFF) in (ord("q"), 27):
            break

    grab.stop()
    cv2.destroyAllWindows()


def run_video(args, model, device):
    cap = cv2.VideoCapture(args.source)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video {args.source}")
    fps = cap.get(cv2.CAP_PROP_FPS) or args.fps_hint
    roi = FaceROI(use_face=not args.no_face)
    frames = []
    print(f"[info] reading {args.source} @ {fps:.2f} fps ...")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(crop_resize_rgb(frame, roi.get(frame), args.img_size))
    cap.release()
    if len(frames) < args.frame_depth + 1:
        raise RuntimeError("Not enough frames.")
    frames = np.stack(frames, 0)
    print(f"[info] {len(frames)} frames; running model ...")

    bvp = infer_bvp(model, frames, device, args.frame_depth, args.data_type)
    t = np.arange(len(frames)) / fps
    out_ts = t[1:1 + len(bvp)]
    bvp_u, fs_u = resample_uniform(bvp, out_ts, round(fps))

    hr, hr_snr = estimate_hr(bvp_u, fs_u)
    rr, rr_snr = estimate_rr(bvp_u, fs_u, min_sec=min(20, len(bvp_u) / fs_u * 0.9))
    dur = len(bvp_u) / fs_u
    print("\n================ RESULTS ================")
    print(f" duration analyzed : {dur:5.1f} s")
    print(f" HR  : {hr:6.1f} bpm   (SNR {hr_snr:5.1f} dB)" if hr else " HR  : n/a")
    print(f" RR  : {rr:6.1f} rpm   (SNR {rr_snr:5.1f} dB)  [approx]" if rr else " RR  : n/a")
    print(f" note: RR < {args.rr_window}s windows are unreliable; record >= 30s of calm breathing.")
    print("=========================================")

    try:
        matplotlib.use("Agg")
        pulse = bandpass(bvp_u, fs_u, 0.7, 3.0)
        f, p = scipy.signal.periodogram(pulse - pulse.mean(), fs=fs_u,
                                        window="hann", nfft=int(fs_u * 60))
        fig, ax = plt.subplots(2, 1, figsize=(10, 6))
        tt = np.arange(len(pulse)) / fs_u
        ax[0].plot(tt[:int(fs_u * 10)], pulse[:int(fs_u * 10)])
        ax[0].set(title="Recovered pulse (first 10 s)", xlabel="s", ylabel="a.u.")
        band = (f >= 0.7) & (f <= 3.0)
        ax[1].plot(f[band] * 60, p[band])
        if hr:
            ax[1].axvline(hr, color="r", ls="--", label=f"HR {hr:.1f} bpm")
            ax[1].legend()
        ax[1].set(title="Pulse spectrum", xlabel="bpm", ylabel="power")
        fig.tight_layout()
        out = os.path.splitext(args.source)[0] + "_rppg.png"
        fig.savefig(out, dpi=120)
        print(f"[info] saved diagnostic plot -> {out}")
    except Exception as e:
        print(f"[info] matplotlib plot skipped ({e})")

In [ ]:
from types import SimpleNamespace

def main():
    weights = "/Users/saptarshiMT/Downloads/iBVP_EfficientPhys.pth"
    source = "0"

    img_size = 72
    frame_depth = 20
    data_type = "standardized"
    hr_window = 10
    rr_window = 30
    update_sec = 1.0
    fps_hint = 30
    snr_min = 3.0
    no_face = False

    args = SimpleNamespace(
        weights=weights,
        source=source,
        img_size=img_size,
        frame_depth=frame_depth,
        data_type=data_type,
        hr_window=hr_window,
        rr_window=rr_window,
        update_sec=update_sec,
        fps_hint=fps_hint,
        snr_min=snr_min,
        no_face=no_face,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model(weights, device, frame_depth, img_size)

    is_file = not source.isdigit() and os.path.exists(source)
    (run_video if is_file else run_webcam)(args, model, device)


if __name__ == "__main__":
    main()